## 💬 Scouting Report Chat: Streamlit Frontend for Your RAG Agent

This notebook is paired with a simple `Streamlit` app that provides a chat interface to your deployed agent.

### 🧠 What It Does:
- Sends user questions and chat history to your **Databricks Model Serving endpoint**
- Displays markdown-formatted responses and cited sources from the RAG agent
- Includes example questions and a “Clear chat” option for easier testing

### ⚙️ Requirements:
Before launching the app, make sure the following environment variables are set:
- `MODEL_SERVING_ENDPOINT` — name of your deployed agent endpoint
- `DATABRICKS_HOST` — (optional) host URL for your workspace
- `DATABRICKS_TOKEN` — your personal access token

### 🚀 To Run the App:
- Deploy via Databricks Apps!


In [0]:
import yaml
import os

# ⚙️ Set Up Configuration

In [0]:
CATALOG = "alexander_booth"
SCHEMA = "rag_demo"
AGENT_NAME = "scouting_reports_agent"
endpoint_name = f'agents_{CATALOG}-{SCHEMA}-{AGENT_NAME}'

# Our frontend application will hit the model endpoint we deployed.
yaml_app_config = {
    "command": ["streamlit", "run", "main.py"],
    "env": [
        {"name": "MODEL_SERVING_ENDPOINT", "value": endpoint_name},
        {"name": "DATABRICKS_TOKEN", "value": "xxxxxxxxxxxxxx"},
        {"name": "DATABRICKS_HOST", "value": "https://e2-demo-field-eng.cloud.databricks.com"}
    ]
}

try:
    with open('chatbot_app/app.yaml', 'w') as f:
        yaml.dump(yaml_app_config, f)
except:
    print('pass to work on build job')

# Write the Streamlit Application code

In [0]:
%%writefile chatbot_app/main.py
import streamlit as st
import os
import requests

# Environment config
SERVING_ENDPOINT = os.environ["MODEL_SERVING_ENDPOINT"]
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST", "https://e2-demo-field-eng.cloud.databricks.com")
DATABRICKS_TOKEN = os.environ["DATABRICKS_TOKEN"]

# Page config
st.set_page_config(page_title="Scouting Report Chat", page_icon="🔎", layout="wide")
st.title("🔎 Scouting Report Chat")
st.caption("This demo lets you chat with a RAG-powered assistant trained on baseball scouting reports. Answers are based only on indexed reports, and source references are included below each response.")

with st.expander("💡 Example questions", expanded=True):
    examples = [
        "Tell me about Roman Anthony.",
        "What are Andrew Painter’s strengths?",
        "Compare Marcelo Mayer to Jordan Lawlar."
    ]
    for example in examples:
        if st.button(example):
            st.session_state.history.append({"role": "user", "content": example})
            st.rerun()

# Clear button
if st.button("🧹 Clear chat"):
    st.session_state.history = []
    st.rerun()

# Session state for chat history
if "history" not in st.session_state:
    st.session_state.history = []

# Input box
user_input = st.chat_input("Ask about a player...")

# When the user submits a new message
if user_input:
    st.session_state.history.append({"role": "user", "content": user_input})

    with st.spinner("Thinking..."):
        try:
            # Format history for endpoint
            messages = [
                {"role": "user", "content": m["content"]} if m["role"] == "user"
                else {"role": "assistant", "content": m["content"]}
                for m in st.session_state.history
            ]

            payload = {
                "query": user_input,
                "messages": messages
            }

            response = requests.post(
                f"{DATABRICKS_HOST}/serving-endpoints/{SERVING_ENDPOINT}/invocations",
                headers={
                    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
                    "Content-Type": "application/json"
                },
                json={"inputs": payload},
                timeout=30
            )

            response.raise_for_status()
            reply = response.json().get("predictions", ["(No answer returned)"])[0]
        except Exception as e:
            reply = f"ERROR: {e}"

        st.session_state.history.append({"role": "assistant", "content": reply})

# Render full chat history
for entry in st.session_state.history:
    with st.chat_message(entry["role"]):
        st.markdown(entry["content"])


Overwriting chatbot_app/main.py


# Deploy to Databricks Apps

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import (
    App, AppResource, AppResourceServingEndpoint,
    AppResourceServingEndpointServingEndpointPermission, AppDeployment
)

# Initialize Databricks workspace client using environment variables / config
w = WorkspaceClient()

# Name of the Streamlit app that will be deployed in the Databricks workspace
app_name = "abooth-data-app"

# Define the app resource permission:
# Allow the Streamlit app to call the model serving endpoint
serving_endpoint = AppResourceServingEndpoint(
    name=endpoint_name,  # Must match your deployed model serving endpoint
    permission=AppResourceServingEndpointServingEndpointPermission.CAN_QUERY
)

# Wrap the serving endpoint resource as a named resource used in the app
rag_endpoint = AppResource(
    name="rag-endpoint",  # Internal resource name for use in the app
    serving_endpoint=serving_endpoint
)

# Define the Streamlit app metadata and its associated resources
rag_app = App(
    name=app_name,
    description="Your Databricks assistant",  # Shown in Workspace UI
    default_source_code_path=os.path.join(os.getcwd(), 'chatbot_app'),  # Path to main.py folder
    resources=[rag_endpoint]  # Add the model endpoint permission
)

# Create the app in the workspace, or skip if it already exists
try:
    app_details = w.apps.create_and_wait(app=rag_app)
    print(app_details)
except Exception as e:
    if "already exists" in str(e):
        print("App already exists, you can deploy it")
    else:
        raise e


App(name='abooth-data-app', active_deployment=None, app_status=ApplicationStatus(message='App has status: App has not been deployed yet. Run your app by deploying source code', state=<ApplicationState.UNAVAILABLE: 'UNAVAILABLE'>), compute_status=ComputeStatus(message='App compute is running.', state=<ComputeState.ACTIVE: 'ACTIVE'>), create_time='2025-06-17T20:50:50Z', creator='alexander.booth@databricks.com', default_source_code_path='', description='Your Databricks assistant', pending_deployment=None, resources=[AppResource(name='rag-endpoint', description=None, job=None, secret=None, serving_endpoint=AppResourceServingEndpoint(name='agents_alexander_booth-rag_demo-scouting_reports_agent', permission=<AppResourceServingEndpointServingEndpointPermission.CAN_QUERY: 'CAN_QUERY'>), sql_warehouse=None)], service_principal_client_id='df3e4efc-03dc-43a4-b718-6d564b320d0d', service_principal_id=5479267597413541, service_principal_name='app-40zbx9 abooth-data-app', update_time='2025-06-17T20:5

In [0]:
# Trigger Application Deployment
deployment = AppDeployment(
  source_code_path=os.path.join(os.getcwd(), 'chatbot_app')
)

app_details = w.apps.deploy_and_wait(app_name=app_name, app_deployment=deployment)

# Let's access the application
w.apps.get(name=app_name).url

'https://abooth-data-app-1444828305810485.aws.databricksapps.com'